In [7]:
import os, urllib.request
import cv2
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display

os.makedirs('models', exist_ok=True)
BASE = 'https://github.com/opencv/opencv_zoo/raw/main/models/'
FILES = {
    'face_detection_yunet_2023mar.onnx': BASE + 'face_detection_yunet/face_detection_yunet_2023mar.onnx',
    'face_recognition_sface_2021dec.onnx': BASE + 'face_recognition_sface/face_recognition_sface_2021dec.onnx',
}
for name, url in FILES.items():
    path = os.path.join('models', name)
    if not os.path.exists(path) or os.path.getsize(path) < 10000:
        urllib.request.urlretrieve(url, path)

detector = cv2.FaceDetectorYN.create(
    'models/face_detection_yunet_2023mar.onnx', '', (320, 320), score_threshold=0.8)
recognizer = cv2.FaceRecognizerSF.create(
    'models/face_recognition_sface_2021dec.onnx', '')
THRESHOLD = 0.363  # ambang cosine similarity yang direkomendasikan untuk SFace


def get_bytes(uploader):
    value = uploader.value
    items = list(value.values()) if isinstance(value, dict) else list(value)
    if not items:
        return None
    return bytes(items[0]['content'])


def decode(data):
    img = cv2.imdecode(np.frombuffer(data, np.uint8), cv2.IMREAD_COLOR)
    if img is None:
        raise ValueError('Gagal membaca gambar.')
    return img


def detect(img):
    h, w = img.shape[:2]
    detector.setInputSize((w, h))
    _, faces = detector.detect(img)
    if faces is None or len(faces) == 0:
        return None
    return faces[int(np.argmax(faces[:, -1]))]


def embed(img, face):
    return recognizer.feature(recognizer.alignCrop(img, face))


up1 = widgets.FileUpload(accept='image/*', multiple=False, description='Pilih Gambar 1')
up2 = widgets.FileUpload(accept='image/*', multiple=False, description='Pilih Gambar 2')
btn = widgets.Button(description='Bandingkan Wajah', button_style='primary')
out = widgets.Output()


def on_click(_):
    with out:
        out.clear_output(wait=True)
        d1, d2 = get_bytes(up1), get_bytes(up2)
        if d1 is None or d2 is None:
            print('Pilih kedua gambar terlebih dahulu.')
            return
        try:
            img1, img2 = decode(d1), decode(d2)
        except ValueError as e:
            print(e)
            return
        f1, f2 = detect(img1), detect(img2)

        fig, axes = plt.subplots(1, 2, figsize=(10, 5))
        same, color, cosine = None, 'gray', None
        if f1 is not None and f2 is not None:
            cosine = float(recognizer.match(embed(img1, f1), embed(img2, f2), cv2.FaceRecognizerSF_FR_COSINE))
            same = cosine >= THRESHOLD
            color = 'limegreen' if same else 'crimson'

        for ax, img, face, title in ((axes[0], img1, f1, 'Gambar 1'), (axes[1], img2, f2, 'Gambar 2')):
            ax.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
            if face is not None:
                x, y, w, h = [int(v) for v in face[:4]]
                ax.add_patch(plt.Rectangle((x, y), w, h, fill=False, edgecolor=color, linewidth=3))
            ax.set_title(title)
            ax.axis('off')
        plt.tight_layout()
        plt.show()

        if f1 is None or f2 is None:
            missing = [str(i + 1) for i, f in enumerate((f1, f2)) if f is None]
            print('Wajah tidak terdeteksi pada gambar ' + ', '.join(missing) + '.')
            print('Coba foto yang lebih jelas dan menghadap depan.')
            return

        percent = max(0.0, min(1.0, cosine)) * 100
        verdict = 'KEMUNGKINAN ORANG YANG SAMA' if same else 'KEMUNGKINAN ORANG BERBEDA'
        print(verdict)
        print(f'Kemiripan: {percent:.1f}%')
        print(f'Cosine similarity: {cosine:.4f} (ambang batas: {THRESHOLD})')


btn.on_click(on_click)
display(widgets.HBox([up1, up2]), btn, out)


Button(button_style='primary', description='Bandingkan Wajah', style=ButtonStyle())

Output()